# Housing Strand - Preprocessing and Split

**MultimodalAI'26 - Housing Demo**

This notebook applies the core feature engineering requirements from `STRAND_GUIDE.md` and performs a property-level split to avoid leakage.

**Objective**
- Load the merged dataset from Notebook 01.
- Engineer required demo features:
  - `cold_risk`
  - `lag_temp`
  - `co2_missing`
  - `co2_imputed`
  - `day_of_week`
- Split train/test by **property reference** (not row-level).
- Verify there is no property overlap across splits.
- Save processed train/test feature files.

**Role tie-in**
- Evidence Analyst owns these split-integrity checks.
- Governance Lead uses this evidence to justify split-integrity verdict text.

**Starter-kit boundary**
- This notebook creates analysis inputs only; participants still author final submission deliverables.

**Prerequisite**
- `demo/data/processed/merged_demo.csv` should already be created.

**Sections**
1. Setup and load merged data.
2. Engineer features and create property-level split.
3. Leakage check and split summary for governance evidence.
4. Visual split sanity checks for discussion.

## Section 1 - Setup and Load Inputs

This cell imports the preprocessing helpers and loads `merged_demo.csv` produced by Notebook 01.

If this file is missing, run Notebook 01 first.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    DEMO_ROOT = cwd
elif (cwd / "demo" / "src").exists():
    DEMO_ROOT = cwd / "demo"
else:
    DEMO_ROOT = cwd.parent

sys.path.append(str(DEMO_ROOT))

from src.data_pipeline import engineer_features, split_by_property

PROCESSED_DIR = DEMO_ROOT / "data" / "processed"
merged = pd.read_csv(PROCESSED_DIR / "merged_demo.csv")
merged["date"] = pd.to_datetime(merged["date"])
merged.head()

## Section 2 - Feature Engineering and Property-Level Split

This cell applies required derived features from the strand guide and splits by property reference.

Why property-level split matters:
- prevents train/test leakage from same property appearing in both folds.

In [ ]:
feat_df = engineer_features(merged)
train_df, test_df = split_by_property(feat_df, test_size=0.25, seed=42)

train_df.to_csv(PROCESSED_DIR / "train_features.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_features.csv", index=False)

overlap = set(train_df["reference"]) & set(test_df["reference"])
print("train rows:", len(train_df), "| test rows:", len(test_df))
print("train properties:", train_df["reference"].nunique(), "| test properties:", test_df["reference"].nunique())
print("property overlap (must be 0):", len(overlap))

## Section 3 - Leakage Check and Split Summary

This cell prints train/test size, number of properties, and overlap check.

Expected result:
- `property overlap (must be 0): 0`

These checks map directly to the split-integrity evidence requirement.

In [ ]:
summary = {
    "train_cold_risk_rate": round(train_df["cold_risk"].mean(), 3),
    "test_cold_risk_rate": round(test_df["cold_risk"].mean(), 3),
    "train_co2_missing_rate": round(train_df["avgCo2"].isna().mean(), 3),
    "test_co2_missing_rate": round(test_df["avgCo2"].isna().mean(), 3),
}
summary

In [ ]:
split_df = pd.concat([
    train_df.assign(split="train"),
    test_df.assign(split="test"),
], ignore_index=True)

plt.figure(figsize=(7, 3))
sns.countplot(data=split_df, x="property_type", hue="split")
plt.xticks(rotation=20, ha="right")
plt.title("Property-type distribution by split")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 3))
sns.barplot(data=split_df, x="split", y="cold_risk")
plt.ylabel("Cold-risk rate")
plt.title("Cold-risk rate by split")
plt.tight_layout()
plt.show()